# Marker OCR vs. manually corrected ground truth

Thin wrapper around `src.ocr.validation.compare`. The metric logic lives in `src/ocr/validation/`; this notebook just loads the report and renders it for the thesis.

Re-run the underlying comparison with:

```bash
python -m src.ocr.validation.compare \
  --gt   draft_codes/ocr_validation_results/content/output_md/wo-exercises_ground_truth_extract_ciam_tc.md \
  --pred draft_codes/ocr_validation_results/content/output_md/extract_ciam_tc/extract_ciam_tc.md \
  --out  experiments/2026-05-12-marker-validation-ciam-tc/results.json
```

**Scope note.** The ground-truth file has its exercise section stripped because marker's output does not cover that section. The comparison is therefore apples-to-apples; document coverage of the exercises is a separate, qualitative concern.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

# Walk up to the repo root (the directory containing pyproject.toml) and put
# it on sys.path so `from src.ocr.validation...` resolves regardless of
# where the notebook is launched from.
repo_root = Path.cwd().resolve()
while not (repo_root / "pyproject.toml").exists():
    if repo_root.parent == repo_root:
        raise RuntimeError("Could not locate repo root (no pyproject.toml found).")
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from src.ocr.validation.compare import build_report  # noqa: E402

GT_PATH = repo_root / "draft_codes/ocr_validation_results/content/output_md/wo-exercises_ground_truth_extract_ciam_tc.md"
PRED_PATH = repo_root / "draft_codes/ocr_validation_results/content/output_md/extract_ciam_tc/extract_ciam_tc.md"
RESULTS_PATH = repo_root / "experiments/2026-05-12-marker-validation-ciam-tc/results.json"

## 1. Load (or recompute) the report

If the cached `results.json` exists, load it. Otherwise rebuild from the source markdowns.

In [ ]:
if RESULTS_PATH.exists():
    report = json.loads(RESULTS_PATH.read_text(encoding="utf-8"))
    print(f"Loaded cached report from {RESULTS_PATH.relative_to(repo_root)}")
else:
    report = build_report(GT_PATH, PRED_PATH)
    RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
    RESULTS_PATH.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"Computed report and wrote to {RESULTS_PATH.relative_to(repo_root)}")

## 2. Headline numbers

- **Prose CER / WER** — character / word error rate on the document with all equations and markdown markup stripped.
- **Equation detection** — precision / recall / F1 treating each equation as a unit to be detected.
- **Aligned-pair accuracy** — distribution of similarity over the equations that were successfully aligned. `exact_match_rate` counts character-for-character matches after LaTeX normalization.

In [ ]:
text = report["text"]
det = report["equations"]["detection"]
acc = report["equations"]["matched_accuracy"]

print("PROSE")
print(f"  CER:                  {text['cer']:.2%}")
print(f"  WER:                  {text['wer']:.2%}")
print(f"  GT chars / Pred chars: {text['gt_chars']} / {text['pred_chars']}")
print()
print("EQUATION DETECTION")
print(f"  GT equations:         {det['gt_count']}")
print(f"  Pred equations:       {det['pred_count']}")
print(f"  Matched:              {det['matched']}")
print(f"  Missing (recall):     {det['missing']}")
print(f"  Hallucinated (prec.): {det['hallucinated']}")
print(f"  Precision / Recall / F1: {det['precision']:.2%} / {det['recall']:.2%} / {det['f1']:.2%}")
print()
print("ALIGNED-PAIR ACCURACY (over matched equations only)")
print(f"  Exact match rate:     {acc['exact_match_rate']:.2%}")
print(f"  >=0.95 similarity:    {acc['high_similarity_rate']:.2%}")
print(f"  Mean similarity:      {acc['mean_similarity']:.4f}")
print(f"  Median similarity:    {acc['median_similarity']:.4f}")
print(f"  P25 / P75 similarity: {acc['p25_similarity']} / {acc['p75_similarity']}")

## 3. Worst aligned equation pairs

These are matched equation pairs ordered by lowest similarity. They expose marker's systematic OCR error modes (e.g. `\Delta` vs `\Lambda`, `\vec{u}` vs `\overrightarrow{u}`, list bullets being absorbed into equations).

In [ ]:
for pair in report["worst_pairs"]:
    print(f"sim={pair['similarity']:.3f}  (gt #{pair['gt_idx']}, pred #{pair['pred_idx']})")
    print(f"  GT:   {pair['gt']}")
    print(f"  PRED: {pair['pred']}")
    print()

## 4. Sample of unmatched equations

**Missing in pred** — equations in the ground truth that the aligner could not pair with anything in marker's output. These are detection misses.

**Hallucinated in pred** — equations marker emitted that did not pair with any ground-truth equation. These are usually layout artifacts (e.g. an entire sentence wrapped in `$...$` by mistake) rather than fabrications.

In [ ]:
print("MISSING IN PRED (sample):")
for s in report["unmatched_gt_samples"]:
    print(f"  gt #{s['idx']}: {s['eq']}")
print()
print("HALLUCINATED IN PRED (sample):")
for s in report["unmatched_pred_samples"]:
    print(f"  pred #{s['idx']}: {s['eq']}")

# Docling OCR vs. manually corrected ground truth


In [3]:
# 0. Set-up
from __future__ import annotations

import json
import sys
from pathlib import Path

# Walk up to the repo root (the directory containing pyproject.toml) and put
# it on sys.path so `from src.ocr.validation...` resolves regardless of
# where the notebook is launched from.
repo_root = Path.cwd().resolve()
while not (repo_root / "pyproject.toml").exists():
    if repo_root.parent == repo_root:
        raise RuntimeError("Could not locate repo root (no pyproject.toml found).")
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from src.ocr.validation.compare import build_report  # noqa: E402

GT_PATH = repo_root / "draft_codes/ocr_validation_results/content/output_md/wo-exercises_ground_truth_extract_ciam_tc.md"
PRED_PATH = repo_root / "draft_codes/ocr_validation_results/content/output_md/extract_ciam_tc/docling_output.md"
RESULTS_PATH = repo_root / "experiments/2026-05-13-docling-validation-ciam-tc/results.json"

In [4]:
# 1. Load the report
if RESULTS_PATH.exists():
    report = json.loads(RESULTS_PATH.read_text(encoding="utf-8"))
    print(f"Loaded cached report from {RESULTS_PATH.relative_to(repo_root)}")
else:
    report = build_report(GT_PATH, PRED_PATH)
    RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
    RESULTS_PATH.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"Computed report and wrote to {RESULTS_PATH.relative_to(repo_root)}")

Computed report and wrote to experiments\2026-05-13-docling-validation-ciam-tc\results.json


In [5]:
# 2. Headline numbers
text = report["text"]
det = report["equations"]["detection"]
acc = report["equations"]["matched_accuracy"]

print("PROSE")
print(f"  CER:                  {text['cer']:.2%}")
print(f"  WER:                  {text['wer']:.2%}")
print(f"  GT chars / Pred chars: {text['gt_chars']} / {text['pred_chars']}")
print()
print("EQUATION DETECTION")
print(f"  GT equations:         {det['gt_count']}")
print(f"  Pred equations:       {det['pred_count']}")
print(f"  Matched:              {det['matched']}")
print(f"  Missing (recall):     {det['missing']}")
print(f"  Hallucinated (prec.): {det['hallucinated']}")
print(f"  Precision / Recall / F1: {det['precision']:.2%} / {det['recall']:.2%} / {det['f1']:.2%}")
print()
print("ALIGNED-PAIR ACCURACY (over matched equations only)")
print(f"  Exact match rate:     {acc['exact_match_rate']:.2%}")
print(f"  >=0.95 similarity:    {acc['high_similarity_rate']:.2%}")
print(f"  Mean similarity:      {acc['mean_similarity']:.4f}")
print(f"  Median similarity:    {acc['median_similarity']:.4f}")
print(f"  P25 / P75 similarity: {acc['p25_similarity']} / {acc['p75_similarity']}")

PROSE
  CER:                  102.68%
  WER:                  133.10%
  GT chars / Pred chars: 15210 / 30173

EQUATION DETECTION
  GT equations:         408
  Pred equations:       2
  Matched:              1
  Missing (recall):     407
  Hallucinated (prec.): 1
  Precision / Recall / F1: 50.00% / 0.25% / 0.49%

ALIGNED-PAIR ACCURACY (over matched equations only)
  Exact match rate:     0.00%
  >=0.95 similarity:    0.00%
  Mean similarity:      0.3000
  Median similarity:    0.3000
  P25 / P75 similarity: None / None


In [4]:
# 3. Worst aligned equation pairs
for pair in report["worst_pairs"]:
    print(f"sim={pair['similarity']:.3f}  (gt #{pair['gt_idx']}, pred #{pair['pred_idx']})")
    print(f"  GT:   {pair['gt']}")
    print(f"  PRED: {pair['pred']}")
    print()

sim=0.333  (gt #245, pred #217)
  GT:   \vec{u}
  PRED: \overrightarrow{u}

sim=0.353  (gt #301, pred #259)
  GT:   r_{2} \circ r_{1}
  PRED: s_{(AC)} \circ r_{(A, \frac{\pi}{2})}

sim=0.385  (gt #248, pred #219)
  GT:   \vec{u} \neq \vec{0}
  PRED: \overrightarrow{u} \neq \overrightarrow{0}

sim=0.489  (gt #199, pred #180)
  GT:   t_{\vec{u}} \circ s_{\Delta}
  PRED: \mathbf{t}_{\vec{u}} \circ \mathbf{s}_{\Lambda}

sim=0.500  (gt #336, pred #305)
  GT:    A = A'
  PRED: -Si\ A = A'

sim=0.560  (gt #257, pred #226)
  GT:   (\mathfrak{D})
  PRED: \mathcal{L}(\mathfrak{D})

sim=0.600  (gt #292, pred #251)
  GT:   t \circ r = s_{\Delta'} \circ s_{\Delta}
  PRED: t \circ r = s_{\Lambda} \cdot \circ s_{\Lambda}

sim=0.625  (gt #256, pred #225)
  GT:   t \circ r = s_{\Delta} \circ s_{\Delta}
  PRED: t \circ r = s_{\Lambda} \cdot \circ s_{\Lambda}

sim=0.650  (gt #35, pred #35)
  GT:   - 81 \equiv 0 [9]
  PRED: \bullet - 81 \equiv 0 [9]

sim=0.667  (gt #156, pred #145)
  GT:   (\Delta')
  PRE

In [5]:
# 4. Sample of unmatched equations
print("MISSING IN PRED (sample):")
for s in report["unmatched_gt_samples"]:
    print(f"  gt #{s['idx']}: {s['eq']}")
print()
print("HALLUCINATED IN PRED (sample):")
for s in report["unmatched_pred_samples"]:
    print(f"  pred #{s['idx']}: {s['eq']}")

MISSING IN PRED (sample):
  gt #65: a \equiv 12 [25]
  gt #66: b \equiv -2 [25]
  gt #67: a + b \equiv 10 [25]
  gt #79: 7^{2002}
  gt #143: (\Delta')
  gt #145: (\Delta')
  gt #146: s_{\Delta'} \circ s_{\Delta}
  gt #148: (\Delta')
  gt #149: 2\overrightarrow{OO'}
  gt #151: (\Delta')
  gt #154: s_{\Delta'} \circ s_{\Delta}
  gt #176: M_1
  gt #177: t_{\vec{u}}
  gt #178: M_1
  gt #179: s_{\Delta}
  gt #180: M_2
  gt #181: s_{\Delta}
  gt #182: t_{\vec{u}}(M_2) = M'
  gt #184: s_{\Delta}
  gt #205: s_{\Delta'} \circ s_{\Delta} = t_{\overrightarrow{HA}}

HALLUCINATED IN PRED (sample):
  pred #256: \begin{array}{c|ccccccccccccccccccccccccccccccccccc
  pred #257: (BH) \perp (CG)
  pred #258: f = s_{(AD)} \circ s_{(CD)} \circ s_{(BC)} \circ s_{(AB)}
  pred #260: \Gamma_{(D,\frac{\pi}{2})} \circ \Gamma_{(A,\frac{\pi}{2})}
  pred #261: r_{(C,-\frac{\pi}{2})} \circ r_{(A,\frac{\pi}{2})}
  pred #262: \begin{array}{ccc} b) & \mathbf{r}_{(D, \frac{\pi}{2})} \circ \mathbf{r}_{(A, \frac{\pi}{2})}